In [6]:
import librosa
import re
import os
import tqdm
import pandas as pd
from IPython.display import Audio, display
from scipy.io import wavfile

In [7]:
def normalize(text):
    text = text.upper()
    text = re.sub(r"[^A-Z' ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [38]:
if os.path.exists('./metadata/train.csv'):
    train_df = pd.read_csv('./metadata/train.csv')
    dev_df = pd.read_csv('./metadata/dev.csv')
    test_df = pd.read_csv('./metadata/test.csv')
else:
    train_df = pd.read_csv('./raw_data/metadata/train.csv')
    dev_df = pd.read_csv('./raw_data/metadata/dev.csv')
    test_df = pd.read_csv('./raw_data/metadata/test.csv')

    data_root = './raw_data'
    out_folder = './data'
    metadata_folder = './metadata'
    os.makedirs(out_folder, exist_ok=True)
    os.makedirs(metadata_folder, exist_ok=True)

    for data_type, df in zip(['train', 'dev', 'test'], [train_df, dev_df, test_df]):
        full_df = []
        for _, row in tqdm.tqdm(df.iterrows(), total=len(df)):
            utt_id = row['utt_id']
            folder = "-".join(utt_id.split('-')[:3])
            utt_id = "-".join(utt_id.split('-')[3:])
            wav_path = os.path.join(data_root, folder, folder, f"{utt_id}.mp3")
            text = row['text']
            assert os.path.exists(wav_path), f"File {wav_path} does not exist."
            subject_id = row['wav'].split('/')[-2]
            
            out_subj_folder = os.path.join(out_folder, subject_id)
            out_wav_folder = os.path.join(out_subj_folder, 'wav')
            os.makedirs(out_wav_folder, exist_ok=True)
            out_transcript_folder = os.path.join(out_subj_folder, 'transcript')
            os.makedirs(out_transcript_folder, exist_ok=True)
            normalized_text = normalize(text)
            
            with open(os.path.join(out_transcript_folder, f"{utt_id}.txt"), 'w') as f:
                f.write(normalized_text)

            audio, _ = librosa.load(wav_path, sr=16000)        
            out_wav_path = os.path.join(out_wav_folder, f"{utt_id}.wav")
            wavfile.write(out_wav_path, 16000, (audio * 32767).astype('int16'))
            full_df.append({
                'audio_path': os.path.abspath(out_wav_path),
                'duration': len(audio) / 16000,
                'text': normalized_text,
                'subject_id': subject_id,
                'utt_id': utt_id,
                'accent': row['accent'],
            })

        full_df = pd.DataFrame(full_df)
        full_df.to_csv(os.path.join(metadata_folder, f"{data_type}.csv"), index=False)

    train_df = pd.read_csv(os.path.join(metadata_folder, 'train.csv'))
    dev_df = pd.read_csv(os.path.join(metadata_folder, 'dev.csv'))
    test_df = pd.read_csv(os.path.join(metadata_folder, 'test.csv'))

In [39]:
# curate metadata 
ACCENTS_EN = [
    "american",
    "british",
    "australian",
    "indian",
    "canadian",
    "bermudian",
    "scottish",
    "african",
    "irish",
    "newzealand",
    "welsh",
    "malaysian",
    "philippine",
    "singaporean",
    "hongkong",
    "southatlantic",
]
ACCENTS_EN = set(ACCENTS_EN)
current_accents = set(train_df['accent'].unique())

In [40]:
accent_mapping = {
    "us": "american",
    "england": "british",
    "australia": "australian",
    "indian": "indian",
    "canada": "canadian",
    "bermuda": "bermudian",
    "scotland": "scottish",
    "african": "african",
    "ireland": "irish",
    "newzealand": "newzealand",
    "wales": "welsh",
    "malaysia": "malaysian",
    "philippines": "philippine",
    "singapore": "singaporean",
    "hongkong": "hongkong",
    'southatlandtic': 'southatlantic',
}

In [41]:
if not current_accents.issubset(ACCENTS_EN):
    train_df['accent'] = train_df['accent'].apply(lambda x: accent_mapping.get(x, x))
    dev_df['accent'] = dev_df['accent'].apply(lambda x: accent_mapping.get(x, x))
    test_df['accent'] = test_df['accent'].apply(lambda x: accent_mapping.get(x, x))

    train_df.to_csv(os.path.join(metadata_folder, 'train.csv'), index=False)
    dev_df.to_csv(os.path.join(metadata_folder, 'dev.csv'), index=False)
    test_df.to_csv(os.path.join(metadata_folder, 'test.csv'), index=False)
else:
    print("All accents are already in the desired format.")

All accents are already in the desired format.
